In [ ]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import os

os.environ["AWS_REGION"] = "us-east-2"

spark = SparkSession.builder \
    .appName("test-gold") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

BUCKET = "clashr-account-data-lake"

# Lê Silver
df_battles = spark.read.parquet(f"s3://{BUCKET}/processed/battles/")
df_profile = spark.read.parquet(f"s3://{BUCKET}/processed/profile/")

print(f"Batalhas Silver: {df_battles.count()} registros")
print(f"Perfil Silver: {df_profile.count()} registros")

In [ ]:
# Cria dim_player
# ─────────────────────────────────────────────

print("\n=== CRIANDO dim_player ===\n")

window_player = Window.partitionBy("player_tag").orderBy(F.col("snapshot_at").desc())

dim_player = df_profile \
    .withColumn("rn", F.row_number().over(window_player)) \
    .filter(F.col("rn") == 1) \
    .drop("rn") \
    .select(
        "player_tag", "player_name", "level",
        "trophies", "best_trophies", "wins",
        "losses", "battle_count", "win_rate",
        "clan_name", "clan_tag", "snapshot_at"
    )

print(f"dim_player: {dim_player.count()} players únicos")
dim_player.show(3)

In [ ]:
# Cria dim_card

print("\n=== CRIANDO dim_card ===\n")

dim_card = df_battles \
    .select(F.explode(F.col("player_deck")).alias("card_name")) \
    .union(
        df_battles.select(F.explode(F.col("opponent_deck")).alias("card_name"))
    ) \
    .distinct() \
    .filter(F.col("card_name").isNotNull())

print(f"dim_card: {dim_card.count()} cartas únicas")
dim_card.show(10)

In [ ]:
# Cria dim_deck

print("\n=== CRIANDO dim_deck ===\n")

dim_deck = df_battles \
    .groupBy("deck_hash", "player_deck") \
    .agg(
        F.count("*").alias("total_battles"),
        F.sum(F.when(F.col("result") == "win", 1).otherwise(0)).alias("wins"),
        F.round(F.avg("player_elixir_avg"), 2).alias("elixir_avg")
    ) \
    .withColumn(
        "win_rate",
        F.round(F.col("wins") / F.col("total_battles") * 100, 2)
    )

print(f"dim_deck: {dim_deck.count()} decks únicos")
print("\n=== TOP 5 DECKS COM MAIOR WIN RATE ===")
dim_deck.orderBy(F.col("win_rate").desc()).show(5)

print("\n=== TOP 5 DECKS MAIS USADOS ===")
dim_deck.orderBy(F.col("total_battles").desc()).show(5)

In [ ]:
# Cria fact_battles

print("\n=== CRIANDO fact_battles ===\n")

fact_battles = df_battles.select(
    "battle_id", "player_tag", "battle_time",
    "battle_type", "result", "crowns_player",
    "crowns_opponent", "player_elixir_avg",
    "deck_hash", "opponent_tag", "ingestion_date"
)

print(f"fact_battles: {fact_battles.count()} batalhas")
print("\n=== AMOSTRA ===")
fact_battles.show(5)

In [ ]:
# Célula 6: Cria metrics_win_rate_by_card

print("\n=== CRIANDO metrics_win_rate_by_card ===\n")

cards_exploded = df_battles.select(
    "player_tag",
    "result",
    F.explode(F.col("player_deck")).alias("card_name")
)

metrics_win_rate_by_card = cards_exploded \
    .groupBy("player_tag", "card_name") \
    .agg(
        F.count("*").alias("appearances"),
        F.sum(F.when(F.col("result") == "win", 1).otherwise(0)).alias("wins")
    ) \
    .withColumn(
        "win_rate",
        F.round(F.col("wins") / F.col("appearances") * 100, 2)
    )

print(f"Total de registros: {metrics_win_rate_by_card.count()}")
print("\n=== TOP 10 CARTAS COM MAIOR WIN RATE ===")
metrics_win_rate_by_card.orderBy(F.col("win_rate").desc()).limit(10).show()

In [ ]:
# Célula 7: Cria metrics_trophy_evolution

print("\n=== CRIANDO metrics_trophy_evolution ===\n")

metrics_trophy_evolution = df_battles \
    .withColumn("week", F.date_trunc("week", F.col("battle_time"))) \
    .groupBy("player_tag", "week") \
    .agg(F.max("player_trophies").alias("trophies")) \
    .orderBy("player_tag", "week")

print(f"Total de registros: {metrics_trophy_evolution.count()}")
print("\n=== EVOLUÇÃO DE TROFÉUS (amostra) ===")
metrics_trophy_evolution.limit(10).show()

In [ ]:
# Célula 8: Cria metrics_top_cards

print("\n=== CRIANDO metrics_top_cards ===\n")

metrics_top_cards = cards_exploded \
    .groupBy("player_tag", "card_name") \
    .agg(
        F.count("*").alias("usage_count"),
        F.sum(F.when(F.col("result") == "win", 1).otherwise(0)).alias("wins")
    ) \
    .withColumn(
        "win_rate",
        F.round(F.col("wins") / F.col("usage_count") * 100, 2)
    ) \
    .orderBy(F.col("usage_count").desc())

print(f"Total de registros: {metrics_top_cards.count()}")
print("\n=== TOP 15 CARTAS MAIS USADAS ===")
metrics_top_cards.limit(15).show()

print("\n\n✅ TODAS AS TRANSFORMAÇÕES SILVER → GOLD VALIDADAS COM SUCESSO!")